<a href="https://colab.research.google.com/github/tillyrcohen-code/python-fundamentals/blob/main/kent_heritage_spatial_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kent Archaeological & Heritage Spatial Intelligence Pipeline
**My Bachelor's Degree Focus:** Classical & Archaeological Studies at the University of Kent  
**Project Goal:** To build an automated data pipeline that absorbs, cleans, stores, and maps archaeological heritage records.

In [11]:
# Import core libraries needed
import sqlite3  # database storage tool
import numpy as np  # random number generator
import pandas as pd  # digital spreadsheet manager

print("Tools imported successfully. Ready to generate data.")

Tools imported successfully. Ready to generate data.


In [12]:
# Set a random seed so the random data looks consistent every time it is run
np.random.seed(42)

# Generate 100 fake artifact records mimicking a regional heritage scheme
raw_data = {
    "artifact_id": [f"KENT-PAS-{i:04d}" for i in range(1, 101)],
    "broadperiod": np.random.choice(
        ["ROMAN", "Roman Period", "IRON AGE", "Iron age", "MEDIEVAL", None],
        100,
        p=[0.35, 0.1, 0.25, 0.05, 0.2, 0.05]
    ),
    "material": np.random.choice(
        ["Copper Alloy", "Bronze", "gold", "Silver", "POTTERY", None],
        100,
        p=[0.3, 0.2, 0.15, 0.15, 0.1, 0.1]
    ),
    "lat": np.random.uniform(51.05, 51.45, 100), # Random coordinates inside Kent
    "lon": np.random.uniform(0.40, 1.40, 100)
}

# Convert raw dictionary into a Pandas DataFrame (a digital table)
df_raw = pd.DataFrame(raw_data)

# Add deliberate missing values into latitude to simulate real messy data
df_raw.loc[df_raw["artifact_id"].sample(4).index, "lat"] = np.nan

print(f"Generated {len(df_raw)} raw archaeological records with simulated messiness.")
print("\nPreview of the messy data:")
df_raw.head()

Generated 100 raw archaeological records with simulated messiness.

Preview of the messy data:


,artifact_id,broadperiod,material,lat,lon
0,KENT-PAS-0001,Roman Period,Copper Alloy,51.306813,0.451682
1,KENT-PAS-0002,None,gold,51.083656,0.931355
2,KENT-PAS-0003,Iron age,Bronze,51.114651,0.940635
3,KENT-PAS-0004,IRON AGE,gold,51.409422,1.037430
4,KENT-PAS-0005,ROMAN,None,51.292572,1.126091


In [13]:
# 1. Make a copy of raw data so it doesn't overwrite the original mess
df_clean = df_raw.copy()

# 2. Drop rows that are missing latitude coordinates
df_clean = df_clean.dropna(subset=["lat"])

# 3. Standardise historical periods (fixing lowercase and mixed spellings)
period_mapping = {
    "ROMAN": "ROMAN",
    "Roman Period": "ROMAN",
    "IRON AGE": "IRON AGE",
    "Iron age": "IRON AGE",
    "MEDIEVAL": "MEDIEVAL"
}
df_clean["broadperiod"] = df_clean["broadperiod"].map(period_mapping).fillna("UNKNOWN")

# 4. Standardise materials
material_mapping = {
    "Copper Alloy": "COPPER ALLOY",
    "Bronze": "COPPER ALLOY",
    "gold": "GOLD",
    "Silver": "SILVER",
    "POTTERY": "CERAMIC"
}
df_clean["material"] = df_clean["material"].map(material_mapping).fillna("OTHER")

print(f"Cleaning complete. Cleaned dataset now contains {len(df_clean)} records.")
print("\nPreview of the newly cleaned data:")
df_clean.head()

Cleaning complete. Cleaned dataset now contains 96 records.

Preview of the newly cleaned data:


,artifact_id,broadperiod,material,lat,lon
0,KENT-PAS-0001,ROMAN,COPPER ALLOY,51.306813,0.451682
1,KENT-PAS-0002,UNKNOWN,GOLD,51.083656,0.931355
2,KENT-PAS-0003,IRON AGE,COPPER ALLOY,51.114651,0.940635
3,KENT-PAS-0004,IRON AGE,GOLD,51.409422,1.037430
4,KENT-PAS-0005,ROMAN,OTHER,51.292572,1.126091


In [14]:
import folium
from folium.plugins import MarkerCluster

# 1. Ensure column names are lowercase
df_clean.columns = [col.lower() for col in df_clean.columns]

# 2. Centre the map over Kent using a free, public Esri tile layer (no watermarks!)
kent_centre = [51.2787, 0.5217]
m = folium.Map(
    location=kent_centre,
    zoom_start=9,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri"
)

# 3. Add a marker cluster plugin
marker_cluster = MarkerCluster().add_to(m)

# 4. Define colour coding for different historical periods
colour_map = {
    "ROMAN": "red",
    "IRON AGE": "orange",
    "MEDIEVAL": "blue",
    "UNKNOWN": "gray"
}

# 5. Loop through each clean record and add a pin to the map
for idx, row in df_clean.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        popup=f"<b>ID:</b> {row['artifact_id']}<br><b>Period:</b> {row['broadperiod']}<br><b>Material:</b> {row['material']}",
        colour=colour_map.get(row["broadperiod"], "black"),
        fill=True,
        fill_opacity=0.8
    ).add_to(marker_cluster)

print("Interactive geospatial map generated successfully!")
# Display the clean map
m


Interactive geospatial map generated successfully!


In [15]:
import folium
from folium.plugins import MarkerCluster

# 1. Ensure column names are lowercase
df_clean.columns = [col.lower() for col in df_clean.columns]

# 2. Centre the map over Kent using the free Esri tile layer
kent_centre = [51.2787, 0.5217]
m = folium.Map(
    location=kent_centre,
    zoom_start=9,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri"
)

# 3. Add a marker cluster plugin
marker_cluster = MarkerCluster().add_to(m)

# 4. Define colour coding for different historical periods
colour_map = {
    "ROMAN": "red",
    "IRON AGE": "orange",
    "MEDIEVAL": "blue",
    "UNKNOWN": "gray"
}

# 5. Loop through each clean record and add interactive pins
for idx, row in df_clean.iterrows():
    # Styled HTML content for the click popup
    popup_html = f"""
    <div style="font-family: sans-serif; font-size: 13px; line-height: 1.4;">
        <b style="color: #333;">Artifact ID:</b> {row['artifact_id']}<br>
        <b>Period:</b> {row['broadperiod']}<br>
        <b>Material:</b> {row['material']}<br>
        <b>Coordinates:</b> {row['lat']:.4f}, {row['lon']:.4f}
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['artifact_id']} ({row['broadperiod']}) - Click for details",
        colour=colour_map.get(row["broadperiod"], "black"),
        fill=True,
        fill_opacity=0.8
    ).add_to(marker_cluster)

print("Interactive map generated successfully!")
# Display the map
m

Interactive map generated successfully!
